# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:** Udaiveer Singh  
**`Roll Number`:** U20230017  
**`GitHub Branch`:** udaiveer_U20230017  

# Imports and Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

# Load Datasets

In [2]:
train_users = pd.read_csv("data/train_users.csv")

print(train_users.head())
print(train_users.shape)

  user_id   age  income  clicks  purchase_amount  session_duration  \
0   U7392   NaN   23053      10           500.00             17.34   
1   U2702  56.0   20239      11           913.33             22.22   
2   U2461   NaN   13907       9          1252.62             41.57   
3   U7475   NaN   26615      12           500.00             30.17   
4   U6040  32.0   27958      13           500.00             65.27   

   content_variety  engagement_score  num_transactions  avg_monthly_spend  \
0          0.36661          37.29781                 3             187.44   
1          0.61370          59.36342                 5             145.15   
2          0.80368          76.78706                 7             282.03   
3          0.26499          30.19441                10             195.35   
4          0.36385          37.12153                 5             439.68   

   ...  screen_brightness  battery_percentage  cart_abandonment_count  \
0  ...                4.0                 2

In [3]:
train_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 33 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   user_id                      2000 non-null   object 
 1   age                          1302 non-null   float64
 2   income                       2000 non-null   int64  
 3   clicks                       2000 non-null   int64  
 4   purchase_amount              2000 non-null   float64
 5   session_duration             2000 non-null   float64
 6   content_variety              2000 non-null   float64
 7   engagement_score             2000 non-null   float64
 8   num_transactions             2000 non-null   int64  
 9   avg_monthly_spend            2000 non-null   float64
 10  avg_cart_value               2000 non-null   float64
 11  browsing_depth               2000 non-null   int64  
 12  revisit_rate                 2000 non-null   float64
 13  scroll_activity   

In [4]:
train_users.describe(include="all")

,user_id,age,income,clicks,purchase_amount,session_duration,content_variety,engagement_score,num_transactions,avg_monthly_spend,...,screen_brightness,battery_percentage,cart_abandonment_count,browser_version,background_app_count,session_inactivity_duration,network_jitter,region_code,subscriber,label
count,2000,1302.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,...,2000.000000,2000.000000,2000.00000,2000,2000.000000,2000.000000,2000.000000,2000,2000,2000
unique,1804,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1973,NaN,NaN,NaN,1284,2,3
top,U8400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,5.93.33,NaN,NaN,NaN,X123,False,user_2
freq,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2,NaN,NaN,NaN,150,1444,712
mean,NaN,39.227343,21564.393500,10.526000,854.920700,32.641035,0.491309,49.325532,6.019000,359.416645,...,3.057850,51.219000,5.05300,NaN,9.456500,15.091855,24.514500,NaN,NaN,NaN
std,NaN,11.407103,6740.282694,3.299207,833.770472,22.769625,0.244320,24.584840,2.561429,413.886399,...,1.102046,27.704671,2.28797,NaN,3.457034,8.547876,18.501836,NaN,NaN,NaN
min,NaN,18.000000,10000.000000,1.000000,500.000000,0.000000,0.005040,0.000000,1.000000,0.000000,...,1.200000,2.000000,0.00000,NaN,4.000000,0.000000,0.000000,NaN,NaN,NaN
25%,NaN,31.000000,16742.500000,8.000000,500.000000,10.987500,0.289365,31.912582,4.000000,150.147500,...,2.100000,28.000000,3.00000,NaN,6.000000,7.657500,10.000000,NaN,NaN,NaN
50%,NaN,39.000000,21107.500000,10.000000,500.000000,33.475000,0.493505,49.566750,6.000000,231.370000,...,3.100000,52.000000,5.00000,NaN,9.000000,15.420000,21.000000,NaN,NaN,NaN
75%,NaN,47.000000,25892.500000,13.000000,596.295000,48.732500,0.661632,64.603265,8.000000,393.437500,...,4.000000,74.000000,6.00000,NaN,12.000000,22.295000,35.000000,NaN,NaN,NaN


## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [5]:
print(train_users.isna().sum())

user_id                          0
age                            698
income                           0
clicks                           0
purchase_amount                  0
session_duration                 0
content_variety                  0
engagement_score                 0
num_transactions                 0
avg_monthly_spend                0
avg_cart_value                   0
browsing_depth                   0
revisit_rate                     0
scroll_activity                  0
time_on_site                     0
interaction_count                0
preferred_price_range            0
discount_usage_rate              0
wishlist_size                    0
product_views                    0
repeat_purchase_gap (days)       0
churn_risk_score                 0
loyalty_index                    0
screen_brightness                0
battery_percentage               0
cart_abandonment_count           0
browser_version                  0
background_app_count             0
session_inactivity_d

In [6]:
# Handle missing values
train_users["age"] = train_users["age"].fillna(train_users["age"].median())

print(train_users.isna().sum())

user_id                        0
age                            0
income                         0
clicks                         0
purchase_amount                0
session_duration               0
content_variety                0
engagement_score               0
num_transactions               0
avg_monthly_spend              0
avg_cart_value                 0
browsing_depth                 0
revisit_rate                   0
scroll_activity                0
time_on_site                   0
interaction_count              0
preferred_price_range          0
discount_usage_rate            0
wishlist_size                  0
product_views                  0
repeat_purchase_gap (days)     0
churn_risk_score               0
loyalty_index                  0
screen_brightness              0
battery_percentage             0
cart_abandonment_count         0
browser_version                0
background_app_count           0
session_inactivity_duration    0
network_jitter                 0
region_cod

In [7]:
# Drop non-informative identifier
train_users = train_users.drop(columns=["user_id"])

## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [8]:
X = train_users.drop(columns=["label"])
y = train_users["label"]

print(X.shape, y.shape)

(2000, 31) (2000,)


In [9]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(label_encoder.classes_)

['user_1' 'user_2' 'user_3']


In [10]:
cat_cols = X.select_dtypes(include=["object", "bool"]).columns
cat_cols

Index(['browser_version', 'region_code', 'subscriber'], dtype='object')

In [11]:
X_encoded = X.copy()

for col in cat_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col])

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [13]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

c:\Anaconda Navigator\Anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [14]:
from sklearn.metrics import accuracy_score, classification_report

y_val_pred = clf.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=label_encoder.classes_
    )
)

Validation Accuracy: 0.8275
              precision    recall  f1-score   support

      user_1       0.86      0.80      0.83       142
      user_2       0.94      0.80      0.86       142
      user_3       0.71      0.90      0.79       116

    accuracy                           0.83       400
   macro avg       0.84      0.83      0.83       400
weighted avg       0.84      0.83      0.83       400



# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
